# 📊 BÁO CÁO Khảo sát Dữ liệu (Master EDA)
Notebook này được thiết kế chuẩn xác theo **12 bước luận điểm EDA** để đưa ra quyết định xử lý Feature và làm sạch dữ liệu.
Tập dữ liệu được sử dụng là toàn bộ **6,825 listings** (Bao gồm cả tin hết hạn) vì giá niêm yết của tin hết hạn vẫn hoàn toàn hợp lệ để huấn luyện mô hình Regression.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Đọc toàn bộ 6,825 listings (File gốc chưa qua cắt Outlier)
df = pd.read_csv("data/phongtro_hanoi_all.csv")
print("Kích thước ban đầu:", df.shape)


## 1. Kiểm tra Tổng quan Dataset
Mục tiêu: Nắm được số dòng, số cột, kiểu dữ liệu và tỷ lệ missing values.


In [ ]:
df.info()
display(df.describe())
display(df.isna().sum().sort_values(ascending=False))


## 2. Phân phối Target (Price) & Outliers
**Quyết định:** Cần tìm xem có giá 100 triệu hay 2 VNĐ không để cắt bỏ, và xem có cần log transform không.


In [ ]:
# Loại bỏ giá trị âm hoặc 0 trước khi vẽ
valid_price = df[df['price'] > 0]['price']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(valid_price, bins=50, kde=True, ax=axes[0])
axes[0].set_title("Histogram - Phân phối Giá gốc")
axes[0].ticklabel_format(style='plain', axis='x')

sns.boxplot(x=valid_price, ax=axes[1])
axes[1].set_title("Boxplot - Phát hiện Outlier của Giá")
axes[1].ticklabel_format(style='plain', axis='x')
plt.show()


## 3. Phân phối Area & Outliers
**Quyết định:** Diện tích 359m2 hay 1m2 có tồn tại không? Cần cắt IQR.


In [ ]:
valid_area = df[df['area'] > 0]['area']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(valid_area, bins=50, kde=True, ax=axes[0])
axes[0].set_title("Histogram - Phân phối Diện tích")

sns.boxplot(x=valid_area, ax=axes[1])
axes[1].set_title("Boxplot - Phát hiện Outlier của Diện tích")
plt.show()


## 4. Quan hệ cốt lõi: Area vs Price
**Mục tiêu:** Diện tích tăng thì giá có tăng tuyến tính không?


In [ ]:
# Lọc nhanh các giá trị hợp lý để scatter plot dễ nhìn
df_clean = df[(df['price'] > 500000) & (df['price'] < 20000000) & (df['area'] > 5) & (df['area'] < 100)]

plt.figure(figsize=(10, 6))
sns.scatterplot(x="area", y="price", data=df_clean, alpha=0.5)
plt.title("Scatter Plot: Diện tích (Area) vs Giá thuê (Price)")
plt.ticklabel_format(style='plain', axis='y')
plt.show()


## 5. Giá thuê theo Quận (District)
**Mục tiêu:** Quận nào đắt nhất? Có nên giữ District làm feature không? Dùng Median thay vì Mean.


In [ ]:
plt.figure(figsize=(14, 6))
# Tính median price theo quận và sắp xếp
order = df_clean.groupby("district")["price"].median().sort_values(ascending=False).index

sns.boxplot(x="district", y="price", data=df_clean, order=order)
plt.title("Boxplot: Phân bổ giá thuê theo Quận (Sắp xếp theo Median)")
plt.xticks(rotation=45)
plt.ticklabel_format(style='plain', axis='y')
plt.show()


## 6. Khảo sát Tiện ích (Amenities)
**Mục tiêu:** Tiện ích nào quá hiếm (như has_desk) thì loại bỏ.


In [ ]:
amenity_cols = [c for c in df.columns if c.startswith('has_')]
amenity_ratios = df[amenity_cols].mean().sort_values(ascending=False) * 100

plt.figure(figsize=(12, 6))
sns.barplot(x=amenity_ratios.values, y=amenity_ratios.index, palette="viridis")
plt.title("Tỷ lệ xuất hiện của các loại Tiện ích (%)")
plt.xlabel("% Xuất hiện trong tin đăng")
plt.show()


## 7. Ảnh hưởng của Tiện ích lên Giá (Tiện ích vs Price)
**Mục tiêu:** Nhóm có thang máy / điều hòa có giá cao hơn hẳn nhóm không có?


In [ ]:
key_amenities = ['has_air_conditioner', 'has_elevator', 'has_washing_machine', 'has_balcony']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, amenity in zip(axes.flatten(), key_amenities):
    sns.boxplot(x=amenity, y="price", data=df_clean, ax=ax)
    ax.set_title(f"Giá phòng vs {amenity}")
    ax.ticklabel_format(style='plain', axis='y')
plt.tight_layout()
plt.show()


## 8. Mùa vụ (Month) vs Giá
**Mục tiêu:** Tháng 8-10 nhập học giá có cao hơn thật không?


In [ ]:
plt.figure(figsize=(10, 5))
month_median = df_clean.groupby('month')['price'].median().reset_index()
sns.barplot(x='month', y='price', data=month_median, palette="coolwarm")
plt.title("Median Price theo Tháng đăng bài")
plt.ticklabel_format(style='plain', axis='y')
plt.show()


## 9. Correlation Heatmap (Đa cộng tuyến)
**Mục tiêu:** Biến nào tương quan mạnh với target nhất? Các biến độc lập có bị đa cộng tuyến không?


In [ ]:
plt.figure(figsize=(12, 10))
num_cols = df_clean.select_dtypes(include=[np.number]).columns
corr = df_clean[num_cols].corr()

# Lọc các biến tương quan > 0.1 với giá để dễ nhìn
important_cols = corr[abs(corr['price']) > 0.1].index
sns.heatmap(df_clean[important_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap (Top features)")
plt.show()


## 10. Bảng Tổng kết Quyết định (Decision Matrix)
| Phát hiện từ EDA | Quyết định Xử lý Pipeline |
| :--- | :--- |
| **Giá (Price):** Bị lệch phải nặng, có tin 100 triệu | Áp dụng IQR loại bỏ râu, dùng **Log Transform (np.log1p)**. |
| **Diện tích (Area):** Có tin 1m2 và 1000m2 ảo | Dùng **IQR** cắt lọc giữ lại phòng từ 5m2 - 45m2. |
| **Quận (District):** Chênh lệch Median rất rõ (Cầu Giấy > Hà Đông) | **Giữ lại**, One-Hot Encoding làm feature chủ chốt. |
| **Area vs Price:** Tương quan dương rõ rệt | Giữ Area làm feature quan trọng nhất. |
| **Tiện ích:** Điều hòa, Máy giặt đẩy giá lên rất mạnh | **Giữ lại**, Fill Null = 0 (Không nhắc tới = Không có). |
| **Mùa vụ:** Tháng 8, 9, 10 giá Median tăng nhẹ | **Giữ lại** biến `is_peak_season`. |
